# 03 — U-Net Evaluation for Whole-Gland Segmentation

## Objective

This notebook evaluates the best U-Net checkpoint obtained during
training for prostate whole-gland segmentation.

The workflow includes:

1. Environment and path configuration
2. Loading validation and test datasets
3. Loading the best U-Net checkpoint
4. Quantitative evaluation using loss, Dice, positive-slice Dice, and IoU
5. Qualitative visualization of segmentation predictions
6. Saving the final segmentation metrics

The test set is used only for final evaluation and is not used for
model selection or hyperparameter tuning.

## 1. Environment Setup

The project root is added to the Python path so that the evaluation
notebook can import the reusable modules implemented in the `src`
package.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II


## 2. Imports and Evaluation Paths

This section imports the PyTorch components and project modules required
to load the validation and test datasets, restore the best U-Net
checkpoint, compute segmentation metrics, and visualize predictions.

In [2]:
import pandas as pd
import torch
import importlib

from torch.utils.data import DataLoader

In [3]:
import src.data.paths
importlib.reload(src.data.paths)

import src.segmentation.dataset
importlib.reload(src.segmentation.dataset)

import src.segmentation.model
importlib.reload(src.segmentation.model)

import src.segmentation.losses
importlib.reload(src.segmentation.losses)

import src.segmentation.train
importlib.reload(src.segmentation.train)

import src.visualization.visualization
importlib.reload(src.visualization.visualization)

<module 'src.visualization.visualization' from 'c:\\Users\\Luiz Trindade\\Documents\\penguin-adventure\\TCC_II\\src\\visualization\\visualization.py'>

In [4]:
from src.data.paths import (
    PROCESSED_DIR,
    SPLITS_DIR,
    CHECKPOINTS_DIR,
)

from src.segmentation.dataset import (
    ProstateSegmentationDataset,
)

from src.segmentation.model import (
    UNet,
)

from src.segmentation.losses import (
    BCEDiceLoss,
)

from src.segmentation.train import (
    evaluate_model,
)

from src.visualization.visualization import (
    plot_segmentation_prediction,
)

In [5]:
processed_dataset_dir = (
    PROCESSED_DIR
    / "whole_gland_2d"
)

images_dir = (
    processed_dataset_dir
    / "images"
)

masks_dir = (
    processed_dataset_dir
    / "masks"
)

splits_csv = (
    SPLITS_DIR
    / "patient_splits_grouped_v2.csv"
)

checkpoint_path = (
    CHECKPOINTS_DIR
    / "unet_whole_gland_grouped_v2_best.pth"
)

print("Images:", images_dir)
print("Masks:", masks_dir)
print("Splits:", splits_csv)
print("Checkpoint:", checkpoint_path)
# Verificação somente de leitura: não altera os CSVs nem o checkpoint.
split_check = pd.read_csv(splits_csv, dtype={"patient_id": str, "study_id": str})
assert len(split_check) == 1500, "Quantidade inesperada de exames na divisão v2."
assert split_check["study_id"].is_unique, "Há exames duplicados."
assert set(split_check["split"]) == {"train", "val", "test"}, "Rótulos de split inesperados."
assert split_check.groupby("patient_id")["split"].nunique().max() == 1, "Vazamento entre pacientes!"
assert checkpoint_path.is_file(), f"Checkpoint não encontrado: {checkpoint_path}"
print("Divisão v2 verificada: 1.500 exames, sem compartilhamento de pacientes.")
print("Contagens:", split_check["split"].value_counts().to_dict())
print("ATENÇÃO: o nome do checkpoint não comprova qual divisão foi usada no treinamento.")


Images: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\whole_gland_2d\images
Masks: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\whole_gland_2d\masks
Splits: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\processed\splits\patient_splits_grouped_v2.csv
Checkpoint: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\checkpoints\unet_whole_gland_grouped_v2_best.pth
Divisão v2 verificada: 1.500 exames, sem compartilhamento de pacientes.
Contagens: {'train': 1049, 'val': 226, 'test': 225}
ATENÇÃO: o nome do checkpoint não comprova qual divisão foi usada no treinamento.


## 3. Evaluation Datasets

The validation and test subsets are loaded using the same patient-level
partition defined during data preparation.

The validation subset is used to report performance on the data used
during model selection, while the test subset provides the final
evaluation on unseen cases.

No data shuffling is applied during evaluation.

In [6]:
validation_dataset = ProstateSegmentationDataset(
    images_dir=images_dir,
    masks_dir=masks_dir,
    splits_csv=splits_csv,
    split="val",
)

test_dataset = ProstateSegmentationDataset(
    images_dir=images_dir,
    masks_dir=masks_dir,
    splits_csv=splits_csv,
    split="test",
)

print(
    "Validation slices:",
    len(validation_dataset),
)

print(
    "Test slices:",
    len(test_dataset),
)

Validation slices: 4195
Test slices: 4171


In [7]:
validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

print(
    "Validation batches:",
    len(validation_loader),
)

print(
    "Test batches:",
    len(test_loader),
)

Validation batches: 525
Test batches: 522


## 4. Best Model Checkpoint

The U-Net architecture is initialized and the best checkpoint obtained
during training is loaded.

The checkpoint corresponds to the epoch with the lowest validation loss
and is used for all subsequent quantitative and qualitative evaluation.

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = UNet(
    in_channels=1,
    out_channels=1,
).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Device:", device)
print(
    "Checkpoint epoch:",
    checkpoint["epoch"],
)
print(
    "Validation loss:",
    checkpoint["validation_loss"],
)

Device: cuda
Checkpoint epoch: 5
Validation loss: 0.09361514627223923


## 5. Validation Set Evaluation

The best U-Net checkpoint is first evaluated on the validation subset.

Performance is reported using the combined segmentation loss, Dice
coefficient, Dice coefficient restricted to slices containing prostate
tissue, and Intersection over Union (IoU).

In [10]:
criterion = BCEDiceLoss()

validation_metrics = evaluate_model(
    model=model,
    dataloader=validation_loader,
    criterion=criterion,
    device=device,
)

print(
    "Validation loss:",
    validation_metrics["loss"],
)

print(
    "Validation Dice:",
    validation_metrics["dice"],
)

print(
    "Validation positive-slice Dice:",
    validation_metrics["positive_dice"],
)

print(
    "Validation IoU:",
    validation_metrics["iou"],
)

Validation loss: 0.09361514627223923
Validation Dice: 0.8380367926330794
Validation positive-slice Dice: 0.830520266941725
Validation IoU: 0.790937708573682


## 6. Test Set Evaluation

After model selection was completed using the validation subset, the
best checkpoint is evaluated on the independent test subset.

The test set is used only for final performance estimation and does not
influence training or checkpoint selection.

In [11]:
test_metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
)

print(
    "Test loss:",
    test_metrics["loss"],
)

print(
    "Test Dice:",
    test_metrics["dice"],
)

print(
    "Test positive-slice Dice:",
    test_metrics["positive_dice"],
)

print(
    "Test IoU:",
    test_metrics["iou"],
)

Test loss: 0.08969180529822963
Test Dice: 0.8466295309907175
Test positive-slice Dice: 0.8397425745109985
Test IoU: 0.8008751868345272


In [12]:
evaluation_results = pd.DataFrame(
    [
        {
            "split": "validation",
            "loss": validation_metrics["loss"],
            "dice": validation_metrics["dice"],
            "positive_dice": validation_metrics["positive_dice"],
            "iou": validation_metrics["iou"],
        },
        {
            "split": "test",
            "loss": test_metrics["loss"],
            "dice": test_metrics["dice"],
            "positive_dice": test_metrics["positive_dice"],
            "iou": test_metrics["iou"],
        },
    ]
)

evaluation_results

,split,loss,dice,positive_dice,iou
0,validation,0.093615,0.838037,0.830520,0.790938
1,test,0.089692,0.846630,0.839743,0.800875


In [13]:
results_dir = (
    PROJECT_ROOT
    / "results"
    / "segmentation"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

metrics_path = (
    results_dir
    / "unet_whole_gland_grouped_v2_metrics.csv"
)

evaluation_results.to_csv(
    metrics_path,
    index=False,
)

print(
    "Metrics saved to:",
    metrics_path,
)

Metrics saved to: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\segmentation\unet_whole_gland_grouped_v2_metrics.csv


In [14]:
saved_metrics = pd.read_csv(
    metrics_path
)

saved_metrics

,split,loss,dice,positive_dice,iou
0,validation,0.093615,0.838037,0.830520,0.790938
1,test,0.089692,0.846630,0.839743,0.800875


## 10. Segmentation Evaluation Summary — grouped_v2

Os valores atualizados devem ser obtidos diretamente das células de execução,
e serão salvos em `results/segmentation/unet_whole_gland_grouped_v2_metrics.csv`.

Nenhum resultado numérico foi pré-preenchido nesta cópia, para não confundir
saídas históricas com resultados recalculados.
